# Imports and data loading

In [2]:
import os
os.environ['HF_HOME'] = '/scratch/bhanushali.ar/netflix_semantic/hf_cache'

import ast
import json
import random
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv

load_dotenv(dotenv_path='/scratch/bhanushali.ar/netflix_semantic/.env')

df = pd.read_csv('/scratch/bhanushali.ar/netflix_semantic/data/processed/movies_processed.csv')
print(df.shape, torch.cuda.get_device_name(0))

(4391, 11) NVIDIA H200


In [3]:
import json
with open('/scratch/bhanushali.ar/netflix_semantic/data/processed/similar_map.json') as f:
    similar_map = {int(k): v for k, v in json.load(f).items()}

embeddings_base = np.load('/scratch/bhanushali.ar/netflix_semantic/data/processed/embeddings.npy')
print(f"loaded similar movies for {len(similar_map)} movies")
print(f"base embeddings: {embeddings_base.shape}")

loaded similar movies for 4391 movies
base embeddings: (4391, 384)


# Training pairs

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

random.seed(42)

id_to_text = dict(zip(df['id'], df['rich_text']))
id_to_idx = {mid: i for i, mid in enumerate(df['id'])}
all_ids = df['id'].tolist()

positive_pairs = []
for movie_id, similar_ids in similar_map.items():
    for sim_id in similar_ids:
        if sim_id in id_to_text:
            positive_pairs.append((id_to_text[movie_id], id_to_text[sim_id]))

negative_pairs = []
for movie_id in similar_map.keys():
    if movie_id not in id_to_idx:
        continue
    idx = id_to_idx[movie_id]
    sims = cosine_similarity([embeddings_base[idx]], embeddings_base)[0]
    ranked = sims.argsort()[::-1]
    similar_set = set(similar_map.get(movie_id, []))
    hard_negs_added = 0
    for r in ranked[1:100]:
        cand_id = all_ids[r]
        if cand_id not in similar_set and cand_id != movie_id:
            negative_pairs.append((id_to_text[movie_id], id_to_text[cand_id]))
            hard_negs_added += 1
        if hard_negs_added >= len(similar_set):
            break

random.shuffle(negative_pairs)
negative_pairs = negative_pairs[:len(positive_pairs)]

print(f"positive pairs: {len(positive_pairs)}, hard negative pairs: {len(negative_pairs)}")

positive pairs: 20867, hard negative pairs: 20867


# QLoRA

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model_name = "unsloth/Llama-3.2-3B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    num_labels=1,
)
base_model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 1/254 [00:27<1:54:40, 27.19s/it]/home/bhanushali.ar/.conda/envs/netflix-semantic/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 254/254 [02:29<00:00,  1.70it/s]
[transformers] LlamaForSequenceClassification LOAD REPORT from: unsloth/Llama-3.2-3B
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 24,316,928 || all params: 3,237,069,824 || trainable%: 0.7512


# Training

In [6]:
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

class RerankerDataset(Dataset):
    def __init__(self, positive_pairs, negative_pairs, tokenizer, max_length=256):
        self.pairs = [(a, b, 1.0) for a, b in positive_pairs] + [(a, b, 0.0) for a, b in negative_pairs]
        random.shuffle(self.pairs)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        query_text, cand_text, label = self.pairs[idx]
        enc = self.tokenizer(
            query_text, cand_text,
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt'
        )
        return {
            'input_ids': enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.float16),
        }

all_pairs = list(zip(positive_pairs, [1.0]*len(positive_pairs))) + list(zip(negative_pairs, [0.0]*len(negative_pairs)))
random.shuffle(all_pairs)
split = int(len(all_pairs) * 0.9)
train_data = all_pairs[:split]

train_pos = [(a, b) for (a, b), label in train_data if label == 1.0]
train_neg = [(a, b) for (a, b), label in train_data if label == 0.0]

train_dataset = RerankerDataset(train_pos, train_neg, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

eval_movies = [mid for mid in similar_map.keys()
               if mid in id_to_idx and len([s for s in similar_map[mid] if s in id_to_idx]) >= 5]
random.shuffle(eval_movies)
eval_movies = eval_movies[:100]

print(f"train pairs: {len(train_dataset)}, eval query movies: {len(eval_movies)}")

train pairs: 37560, eval query movies: 100


In [7]:
from sklearn.metrics import ndcg_score

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
accumulation_steps = 8
epochs = 3
SAVE_PATH = '/scratch/bhanushali.ar/netflix_semantic/models/finetuned'

def score_pair(query_text, cand_text):
    enc = tokenizer(query_text, cand_text, truncation=True, max_length=256,
                    padding='max_length', return_tensors='pt').to('cuda')
    with torch.no_grad():
        with torch.amp.autocast('cuda'):
            logit = model(**enc).logits.squeeze()
    return torch.sigmoid(logit).item()

def eval_ndcg():
    model.eval()
    ndcgs = []
    for mid in eval_movies:
        idx = id_to_idx[mid]
        # top-50 base-retrieval candidates
        sims = cosine_similarity([embeddings_base[idx]], embeddings_base)[0]
        sims[idx] = -1
        cand_idx = sims.argsort()[::-1][:50]
        query_text = df.iloc[idx]['rich_text']
        # rerank candidates through the model
        scores = []
        for ci in cand_idx:
            s = score_pair(query_text, df.iloc[ci]['rich_text'])
            scores.append(s)
        order = np.argsort(scores)[::-1][:10]
        top10_ids = [all_ids[cand_idx[o]] for o in order]
        gt = set(s for s in similar_map[mid] if s in id_to_idx)
        rel = [1.0 if tid in gt else 0.0 for tid in top10_ids]
        ideal = sorted(rel, reverse=True)
        if sum(ideal) > 0:
            ndcgs.append(ndcg_score([ideal], [rel]))
        else:
            ndcgs.append(0.0)
    return np.mean(ndcgs)

best_ndcg = -1

for epoch in range(epochs):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        with torch.amp.autocast('cuda'):
            logits = model(input_ids=batch['input_ids'].cuda(),
                           attention_mask=batch['attention_mask'].cuda()).logits.squeeze(-1)
            labels = batch['label'].cuda().float()
            loss = F.binary_cross_entropy_with_logits(logits, labels) / accumulation_steps

        loss.backward()
        total_loss += loss.item() * accumulation_steps

        if (step + 1) % accumulation_steps == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()

        if (step + 1) % 500 == 0:
            print(f"epoch {epoch+1} step {step+1}/{len(train_loader)} loss: {total_loss/(step+1):.4f}")

    avg_loss = total_loss / len(train_loader)
    ndcg = eval_ndcg()
    print(f"Epoch {epoch+1}/{epochs} — train_loss: {avg_loss:.4f}, NDCG@10: {ndcg:.4f}")

    if ndcg > best_ndcg:
        best_ndcg = ndcg
        model.save_pretrained(SAVE_PATH)
        tokenizer.save_pretrained(SAVE_PATH)
        print(f"  new best NDCG {ndcg:.4f} — checkpoint saved")
    else:
        print(f"  NDCG {ndcg:.4f} did not beat best {best_ndcg:.4f} — not saving")

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


epoch 1 step 500/4695 loss: 0.6237
epoch 1 step 1000/4695 loss: 0.5271
epoch 1 step 1500/4695 loss: 0.4617
epoch 1 step 2000/4695 loss: 0.4259
epoch 1 step 2500/4695 loss: 0.4054
epoch 1 step 3000/4695 loss: 0.3877
epoch 1 step 3500/4695 loss: 0.3732
epoch 1 step 4000/4695 loss: 0.3651
epoch 1 step 4500/4695 loss: 0.3599
Epoch 1/3 — train_loss: 0.3594, NDCG@10: 0.1692
  new best NDCG 0.1692 — checkpoint saved
epoch 2 step 500/4695 loss: 0.2739
epoch 2 step 1000/4695 loss: 0.2583
epoch 2 step 1500/4695 loss: 0.2617
epoch 2 step 2000/4695 loss: 0.2621
epoch 2 step 2500/4695 loss: 0.2570
epoch 2 step 3000/4695 loss: 0.2550
epoch 2 step 3500/4695 loss: 0.2517
epoch 2 step 4000/4695 loss: 0.2468
epoch 2 step 4500/4695 loss: 0.2462
Epoch 2/3 — train_loss: 0.2453, NDCG@10: 0.2039
  new best NDCG 0.2039 — checkpoint saved
epoch 3 step 500/4695 loss: 0.2534
epoch 3 step 1000/4695 loss: nan
epoch 3 step 1500/4695 loss: nan
epoch 3 step 2000/4695 loss: nan
epoch 3 step 2500/4695 loss: nan
epoch 3